<a href="https://colab.research.google.com/github/aryamantepal/cuda-transformer/blob/main/OptimizedTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# imports

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

In [6]:
!pip install lightning

import lightning as L
from torch.profiler import schedule, tensorboard_trace_handler
from lightning.pytorch.profilers import PyTorchProfiler

In [7]:
class PositionEncoding(nn.Module):

    def __init__(self, d_model=2, max_len=6):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1)
        embedding_index = torch.arange(start=0, end=d_model, step=2).float()

        div_term = 1/torch.tensor(10000.0)**(embedding_index / d_model)


        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)


    def forward(self, word_embeddings):
        T = word_embeddings.size(1)
        return word_embeddings + self.pe[:T, :].unsqueeze(0)

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        # Project then reshape into heads: (B, T, D) -> (B, H, T, head_dim)
        q = self.W_q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # (B, H, T, T) — this is the matrix that explodes in size with seq_len
        sims = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask is (T, T), broadcast to (1, 1, T, T) for all batches and heads
            sims = sims.masked_fill(mask.unsqueeze(0).unsqueeze(0), -1e9)

        attn_probs = F.softmax(sims, dim=-1)
        attn_out = torch.matmul(attn_probs, v)  # (B, H, T, head_dim)

        # Recombine heads: (B, H, T, head_dim) -> (B, T, D)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(attn_out)

In [9]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        # d_ff = 4 * d_model is standard — this is where most FLOPs live
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x, mask=None):
        # Pre-norm style (more stable than post-norm at scale)
        x = x + self.attn(self.norm1(x), mask=mask)
        x = x + self.ffn(self.norm2(x))
        return x

In [10]:
class DecoderOnlyTransformer(L.LightningModule):
    def __init__(self, num_tokens=32000, d_model=512, num_heads=8,
                 num_layers=6, d_ff=2048, max_len=512):
        super().__init__()
        L.seed_everything(seed=42)

        self.we = nn.Embedding(num_tokens, d_model)
        self.pe = PositionEncoding(d_model=d_model, max_len=max_len)
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.fc_layer = nn.Linear(d_model, num_tokens)
        self.loss = nn.CrossEntropyLoss()

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.we(token_ids)       # (B, T, D)
        x = self.pe(x)

        mask = ~torch.tril(torch.ones(T, T, device=token_ids.device)).bool()

        for layer in self.layers:
            x = layer(x, mask=mask)

        x = self.norm(x)
        return self.fc_layer(x)      # (B, T, V)

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=3e-4)

    def training_step(self, batch, batch_idx):
        input_tokens, labels = batch
        logits = self.forward(input_tokens)
        loss = self.loss(logits.view(-1, logits.size(-1)), labels.view(-1))
        return loss

In [11]:
device = torch.device("cuda")

model = DecoderOnlyTransformer(
    num_tokens=32000,
    d_model=512,
    num_heads=8,
    num_layers=6,
    d_ff=2048,
    max_len=512,
).to(device)

num_samples = 256
seq_len = 512
batch_size = 16

input_data = torch.randint(0, 32000, (num_samples, seq_len))
label_data = torch.randint(0, 32000, (num_samples, seq_len))

dataset = TensorDataset(input_data, label_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

profiler = PyTorchProfiler(
    schedule=schedule(wait=1, warmup=2, active=3, repeat=1),
    on_trace_ready=tensorboard_trace_handler("lightning_logs/profiler"),
    record_shapes=True,
    profile_memory=True,
)

trainer = L.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    profiler=profiler,
    enable_checkpointing=False,
    logger=False,
)

trainer.fit(model, dataloader)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cu

┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ we       │ Embedding        │ 16.4 M │ train │     0 │
│ 1 │ pe       │ PositionEncoding │      0 │ train │     0 │
│ 2 │ layers   │ ModuleList       │ 18.9 M │ train │     0 │
│ 3 │ norm     │ LayerNorm        │  1.0 K │ train │     0 │
│ 4 │ fc_layer │ Linear           │ 16.4 M │ train │     0 │
│ 5 │ loss     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 51.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 51.7 M                                                                                               
Total estimated model params size (MB): 206                                                                        
Modules in train mode: 78                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


INFO: FIT Profiler Report
Profile stats for: records
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     272.929ms        49.38%     272.929ms      90.976ms  

In [13]:
# Run this AFTER trainer.fit to get a memory timeline
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |   5947 MiB | 267700 MiB | 267700 MiB |
|       from large pool |      0 B   |   5871 MiB | 266784 MiB | 266784 MiB |
|       from small pool |      0 B   |    123 MiB |    915 MiB |    915 MiB |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |   5947 MiB | 267700 MiB | 267700 MiB |
|       from large pool |      0 B   |   5871 MiB | 266784 MiB |